In [13]:
import pandas as pd
import os
import logging

Run_name = 'Run_285'

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tropomi_merging.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

def merge_and_save_dataframes(Run_name):
    # Path to the original file with t2m data
    original_file_path = '/net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_hourly_emissions_with_qa.csv'
    
    # Path to the new processed data
    new_file_path = f'../data/{Run_name}/valid_tropomi_emissions_with_qa.csv'
    
    # Output directory and filename
    output_dir = os.path.dirname(new_file_path)
    output_path = os.path.join(output_dir, 'updated_tropomi_hourly_emissions.csv')

    logger.info(f"Loading original data from: {original_file_path}")
    original_df = pd.read_csv(original_file_path)
    logger.info(f"Loaded {len(original_df)} rows from original data")
    
    logger.info(f"Loading new processed data from: {new_file_path}")
    new_df = pd.read_csv(new_file_path)
    logger.info(f"Loaded {len(new_df)} rows from new processed data")
    
    # Create a merged dataframe based on identifying columns
    # Assuming location, latitude, longitude, and utc_time can uniquely identify a row
    logger.info("Creating mapping between original and new data")
    
    # Create matching keys in both dataframes
    original_df['match_key'] = original_df['location'].astype(str) + '_' + \
                               original_df['latitude'].astype(str) + '_' + \
                               original_df['longitude'].astype(str) + '_' + \
                               original_df['utc_time'].astype(str)
    
    new_df['match_key'] = new_df['location'].astype(str) + '_' + \
                          new_df['latitude'].astype(str) + '_' + \
                          new_df['longitude'].astype(str) + '_' + \
                          new_df['utc_time'].astype(str)
    
    # Create a dictionary mapping from match_key to new values
    new_values_dict = {}
    for idx, row in new_df.iterrows():
        new_values_dict[row['match_key']] = {
            'plume_label': row['plume_label'],
            # 'no2_std_50km': row['no2_std_50km'], 
            # Add any other columns you want to update here
        }
    
    # Count matches for reporting
    match_count = 0
    
    # Update the original dataframe with new values
    logger.info("Updating original dataframe with new values")
    for idx, row in original_df.iterrows():
        match_key = row['match_key']
        if match_key in new_values_dict:
            match_count += 1
            original_df.at[idx, 'plume_label'] = new_values_dict[match_key]['plume_label']
            # original_df.at[idx, 'no2_std_50km'] = new_values_dict[match_key]['no2_std_50km']
            # Update any other columns here
    
    logger.info(f"Updated {match_count} rows out of {len(original_df)} original rows")
    
    # Drop the no2_var_50km column and the temporary match_key column
    logger.info("Dropping unwanted columns")
    columns_to_drop = ['no2_var_50km', 'match_key']
    original_df = original_df.drop(columns=columns_to_drop, errors='ignore')
    original_df = original_df.dropna()
    
    # Save the updated dataframe
    logger.info(f"Saving updated dataframe to: {output_path}")
    original_df.to_csv(output_path, index=False)
    logger.info(f"Successfully saved {len(original_df)} rows to {output_path}")
    
    return output_path

if __name__ == "__main__":
    logger.info("=" * 80)
    logger.info("Starting TROPOMI data merge")
    logger.info("=" * 80)
    
    output_file = merge_and_save_dataframes(Run_name)
    
    logger.info("=" * 80)
    logger.info(f"Merge completed. Output saved to: {output_file}")
    logger.info("=" * 80)

2025-05-22 22:50:19,720 - INFO - ================================================================================
2025-05-22 22:50:19,721 - INFO - Starting TROPOMI data merge
2025-05-22 22:50:19,722 - INFO - ================================================================================
2025-05-22 22:50:19,723 - INFO - Loading original data from: /net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_hourly_emissions_with_qa.csv
2025-05-22 22:50:24,983 - INFO - Loaded 501753 rows from original data
2025-05-22 22:50:24,984 - INFO - Loading new processed data from: ../data/Run_285/valid_tropomi_emissions_with_qa.csv
2025-05-22 22:50:26,953 - INFO - Loaded 639901 rows from new processed data
2025-05-22 22:50:26,954 - INFO - Creating mapping between original and new data
2025-05-22 22:50:57,095 - INFO - Updating original dataframe with new values
2025-05-22 22:51:27,629 - INFO - Updated 501753 rows out of 501753 original rows
2025-05-22 22:51:27,630 - INFO - Dropping unwanted column

In [1]:
import xarray as xr
ds = xr.open_dataset('/net/fs06/d3/rzhuang/TROPOMI_US/data/era5/TOA_incident_solar_radiation.grib', engine='cfgrib')
ds.to_netcdf('/net/fs06/d3/rzhuang/TROPOMI_US/data/era5/TOA_incident_solar_radiation.nc')

/net/fs01/home/rzhuang/miniforge3/lib/python3.12/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [2]:
import xarray as xr
ds = xr.open_dataset('/net/fs06/d3/rzhuang/TROPOMI_US/data/era5/Total_column_water_vapour.grib', engine='cfgrib')
ds.to_netcdf('/net/fs06/d3/rzhuang/TROPOMI_US/data/era5/Total_column_water_vapour.nc')

/net/fs01/home/rzhuang/miniforge3/lib/python3.12/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [7]:
Run_name = 'Run_446'
tropomi_combined_dropna = pd.read_csv(
    f'/net/fs06/d3/rzhuang/TROPOMI_US/data/{Run_name}/updated_tropomi_hourly_emissions.csv',
)
tropomi_combined_dropna.columns

Index(['location', 'facility_id', 'latitude', 'longitude', 'utc_time',
       'wind_u', 'wind_v', 'plume_label', 'file_path', 'State',
       'annual_nox_emission', 'no2_mean_50km', 'no2_frac_valid_50km',
       'surface_altitude', 'surface_altitude_precision',
       'surface_classification', 'surface_pressure', 'surface_albedo',
       'surface_albedo_nitrogendioxide_window', 'cloud_pressure_crb',
       'cloud_fraction_crb', 'cloud_albedo_crb', 'scene_albedo',
       'apparent_scene_pressure', 'snow_ice_flag', 'aerosol_index_354_388',
       'eastward_wind', 'northward_wind', 'scaled_small_pixel_variance',
       'tropospheric_NO2_column_number_density', 'sensor_altitude',
       'sensor_azimuth_angle', 'sensor_zenith_angle', 'solar_azimuth_angle',
       'solar_zenith_angle', 'nearby_plants_count_20km', 'total_emission_20km',
       'percentage_emission_20km', 'nearby_plants_count_50km',
       'total_emission_50km', 'percentage_emission_50km',
       'nearby_plants_count_100km', '

In [ ]:
import pandas as pd
import os
import logging

# load the full power_plants DF
power_plants = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/power_plants_with_combined_nearby_stats_parallel_debug.csv')

# filter: no other plants within 20 km, but at least one city within 20 km
mask = (
    (power_plants['nearby_plants_count_20km'] == 0) &
    (power_plants['nearby_cities_count_20km'] == 0)
)

# apply filter and take the first 100 (or, if you have a ranking metric, sort before .head)
top100 = power_plants.loc[mask].head(100)
top50 = power_plants.loc[mask].head(50)
top20 = power_plants.loc[mask].head(20)

tropomi_combined_dropna = pd.read_csv(
    f'/net/fs06/d3/rzhuang/TROPOMI_US/data/{Run_name}/updated_tropomi_hourly_emissions.csv',
)

filtered_100 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top100['Facility ID'])]
filtered_50 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top50['Facility ID'])]
filtered_20 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top20['Facility ID'])]

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = tropomi_combined_dropna.iloc[:,10:]
# X = df.loc['surface_albedo']

y = tropomi_combined_dropna["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.7976303175852757
Precision: 0.40283177812338
Recall: 0.8688329839273236
F1 Score: 0.5504493735334485
Confusion Matrix:
 [[67610 18431]
 [ 1877 12433]]
AUC: 0.912814526830289
Classification Report:
               precision    recall  f1-score   support

       False       0.97      0.79      0.87     86041
        True       0.40      0.87      0.55     14310

    accuracy                           0.80    100351
   macro avg       0.69      0.83      0.71    100351
weighted avg       0.89      0.80      0.82    100351

Kappa: 0.4416525783491849


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_100.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_100["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.8124725323020129
Precision: 0.639776171601772
Recall: 0.8234058514628657
F1 Score: 0.7200682280390999
Confusion Matrix:
 [[12999  3090]
 [ 1177  5488]]
AUC: 0.9026561507055861
Classification Report:
               precision    recall  f1-score   support

       False       0.92      0.81      0.86     16089
        True       0.64      0.82      0.72      6665

    accuracy                           0.81     22754
   macro avg       0.78      0.82      0.79     22754
weighted avg       0.84      0.81      0.82     22754

Kappa: 0.5823930221277251


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_50.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_50["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.816297802455619
Precision: 0.7360968952575913
Recall: 0.8432675395739692
F1 Score: 0.7860460879861554
Confusion Matrix:
 [[6123 1547]
 [ 802 4315]]
AUC: 0.9063423580523444
Classification Report:
               precision    recall  f1-score   support

       False       0.88      0.80      0.84      7670
        True       0.74      0.84      0.79      5117

    accuracy                           0.82     12787
   macro avg       0.81      0.82      0.81     12787
weighted avg       0.82      0.82      0.82     12787

Kappa: 0.6263946316745456


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_20.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_20["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.8549469964664311
Precision: 0.8426966292134831
Recall: 0.8808290155440415
F1 Score: 0.8613409896976862
Confusion Matrix:
 [[2289  476]
 [ 345 2550]]
AUC: 0.9364862408530016
Classification Report:
               precision    recall  f1-score   support

       False       0.87      0.83      0.85      2765
        True       0.84      0.88      0.86      2895

    accuracy                           0.85      5660
   macro avg       0.86      0.85      0.85      5660
weighted avg       0.86      0.85      0.85      5660

Kappa: 0.7094317772530815


In [ ]:
import pandas as pd
import os
import logging

Run_name = 'Run_285'

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tropomi_merging.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

def merge_and_save_dataframes(Run_name):
    # Path to the original file with t2m data
    original_file_path = '/net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_hourly_emissions_with_qa.csv'
    
    # Path to the new processed data
    new_file_path = f'../data/{Run_name}/valid_tropomi_emissions_with_qa.csv'
    
    # Output directory and filename
    output_dir = os.path.dirname(new_file_path)
    output_path = os.path.join(output_dir, 'updated_tropomi_hourly_emissions.csv')

    logger.info(f"Loading original data from: {original_file_path}")
    original_df = pd.read_csv(original_file_path)
    logger.info(f"Loaded {len(original_df)} rows from original data")
    
    logger.info(f"Loading new processed data from: {new_file_path}")
    new_df = pd.read_csv(new_file_path)
    logger.info(f"Loaded {len(new_df)} rows from new processed data")
    
    # Create a merged dataframe based on identifying columns
    # Assuming location, latitude, longitude, and utc_time can uniquely identify a row
    logger.info("Creating mapping between original and new data")
    
    # Create matching keys in both dataframes
    original_df['match_key'] = original_df['location'].astype(str) + '_' + \
                               original_df['latitude'].astype(str) + '_' + \
                               original_df['longitude'].astype(str) + '_' + \
                               original_df['utc_time'].astype(str)
    
    new_df['match_key'] = new_df['location'].astype(str) + '_' + \
                          new_df['latitude'].astype(str) + '_' + \
                          new_df['longitude'].astype(str) + '_' + \
                          new_df['utc_time'].astype(str)
    
    # Create a dictionary mapping from match_key to new values
    new_values_dict = {}
    for idx, row in new_df.iterrows():
        new_values_dict[row['match_key']] = {
            'plume_label': row['plume_label'],
            # 'no2_std_50km': row['no2_std_50km'], 
            # Add any other columns you want to update here
        }
    
    # Count matches for reporting
    match_count = 0
    
    # Update the original dataframe with new values
    logger.info("Updating original dataframe with new values")
    for idx, row in original_df.iterrows():
        match_key = row['match_key']
        if match_key in new_values_dict:
            match_count += 1
            original_df.at[idx, 'plume_label'] = new_values_dict[match_key]['plume_label']
            # original_df.at[idx, 'no2_std_50km'] = new_values_dict[match_key]['no2_std_50km']
            # Update any other columns here
    
    logger.info(f"Updated {match_count} rows out of {len(original_df)} original rows")
    
    # Drop the no2_var_50km column and the temporary match_key column
    logger.info("Dropping unwanted columns")
    columns_to_drop = ['no2_var_50km', 'match_key']
    original_df = original_df.drop(columns=columns_to_drop, errors='ignore')
    original_df = original_df.dropna()
    
    # Save the updated dataframe
    logger.info(f"Saving updated dataframe to: {output_path}")
    original_df.to_csv(output_path, index=False)
    logger.info(f"Successfully saved {len(original_df)} rows to {output_path}")
    
    return output_path

if __name__ == "__main__":
    logger.info("=" * 80)
    logger.info("Starting TROPOMI data merge")
    logger.info("=" * 80)
    
    output_file = merge_and_save_dataframes(Run_name)
    
    logger.info("=" * 80)
    logger.info(f"Merge completed. Output saved to: {output_file}")
    logger.info("=" * 80)

import pandas as pd
import os
import logging
Run_name = 'Run_283'
# load the full power_plants DF
power_plants = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/power_plants_with_combined_nearby_stats_parallel_debug.csv')

# filter: no other plants within 20 km, but at least one city within 20 km
mask = (
    (power_plants['nearby_plants_count_20km'] == 0) &
    (power_plants['nearby_cities_count_20km'] == 0)
)

# apply filter and take the first 100 (or, if you have a ranking metric, sort before .head)
top100 = power_plants.loc[mask].head(100)
top50 = power_plants.loc[mask].head(50)
top20 = power_plants.loc[mask].head(20)

tropomi_combined_dropna = pd.read_csv(
    f'/net/fs06/d3/rzhuang/TROPOMI_US/data/{Run_name}/updated_tropomi_hourly_emissions.csv',
)

filtered_100 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top100['Facility ID'])]
filtered_50 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top50['Facility ID'])]
filtered_20 = tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(top20['Facility ID'])]

X = tropomi_combined_dropna.iloc[:,10:]
# X = df.loc['surface_albedo']

y = tropomi_combined_dropna["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))
# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_100.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_100["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)
# print(X)
X = filtered_50.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_50["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))

# print(X)
X = filtered_20.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_20["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))